# 18 — Attention: Queries, Keys, Values

**Learning objective.** Compute scaled dot-product attention directly and inspect attention weights as a probability distribution.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## 🧠 Visual engineering mental model

![Causal mindmap](assets/mindmaps/18_attention_mechanism.svg)

Follow the information flow, then ask what the control knob changes before touching code.

## 🎛️ Change map — cause → representation → behavior

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Increase **query-key similarity** | softmax weight rises | that value contributes more to the output |
| Apply a **mask** | some positions become unreachable | information flow is structurally constrained |
| Use multiple heads | different projection subspaces attend independently | model can route several relation types |

**Engineering habit:** make one intervention, state the expected direction of change, then measure it.

## 🔮 Predict before you run

1. If a query exactly matches one key, which value should dominate?
2. What happens to the output when an attention row becomes uniform?

### When to use
Use attention when each position should dynamically choose which other information to mix.

### When not / caution
Do not read an attention heatmap as proof of causal explanation.

### Debugging lens
Inspect pre-softmax scores, scaling, masks and row sums before interpreting weights.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


Scaled dot-product attention:
$$A=\mathrm{softmax}(QK^T/\sqrt{d_k}),\qquad \mathrm{Attention}(Q,K,V)=AV$$

Each query chooses a weighted mixture of value vectors based on query–key compatibility.

In [2]:
import torch
torch.manual_seed(42)
Q=torch.tensor([[1.,0.],[0.,1.]])
K=torch.tensor([[1.,0.],[.8,.2],[0.,1.]])
V=torch.tensor([[10.,0.],[8.,2.],[0.,10.]])
scores=Q@K.T/(Q.shape[-1]**0.5)
weights=torch.softmax(scores,dim=-1)
out=weights@V
print('scores\n',scores)
print('weights\n',weights)
print('row sums:',weights.sum(-1))
print('outputs\n',out)

scores
 tensor([[0.7071, 0.5657, 0.0000],
        [0.0000, 0.1414, 0.7071]])
weights
 tensor([[0.4235, 0.3677, 0.2088],
        [0.2392, 0.2756, 0.4852]])
row sums: tensor([1., 1.])
outputs
 tensor([[7.1765, 2.8235],
        [4.5969, 5.4031]])


In [3]:
assert torch.allclose(weights.sum(-1),torch.ones(2))
print('Invariant passed: every attention row sums to 1.')

Invariant passed: every attention row sums to 1.


In [4]:

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5,3))
im=ax.imshow(weights.numpy(),aspect='auto',vmin=0,vmax=1)
ax.set_xlabel('key/value position'); ax.set_ylabel('query position'); ax.set_title('Attention weights')
ax.set_xticks(range(weights.shape[1])); ax.set_yticks(range(weights.shape[0]))
for i in range(weights.shape[0]):
    for j in range(weights.shape[1]): ax.text(j,i,f'{weights[i,j]:.2f}',ha='center',va='center')
fig.colorbar(im,ax=ax,label='probability'); plt.tight_layout(); plt.show()


[static visualization generated successfully during execution; rerun in Jupyter/VS Code to display]


---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Derive the Q/K/V computation
- Interpret attention weights without assuming they are causal explanations